# Pyomo Examples with ripopt

This notebook demonstrates using ripopt as a Pyomo solver via `SolverFactory('ripopt')`.
We compare results with IPOPT on several standard NLP problems.

In [1]:
import pyomo_ripopt
from pyomo.environ import *

## Example 1: Rosenbrock Function (Unconstrained)

Minimize $f(x,y) = 100(y - x^2)^2 + (1-x)^2$

Known solution: $(x^*, y^*) = (1, 1)$, $f^* = 0$

In [7]:
m = ConcreteModel()
m.x = Var(initialize=-1.2)
m.y = Var(initialize=1.0)
m.obj = Objective(expr=100*(m.y - m.x**2)**2 + (1 - m.x)**2)

# Solve with ripopt
solver = SolverFactory('ripopt')
result = solver.solve(m, tee=True)
print(f"ripopt: {result.solver.termination_condition}")
print(f"  obj = {value(m.obj):.6e}")
print(f"  x = {value(m.x):.10f}, y = {value(m.y):.10f}")

# Solve with ipopt for comparison
m.x.set_value(-1.2)
m.y.set_value(1.0)
solver_ipopt = SolverFactory('ipopt')
result_ipopt = solver_ipopt.solve(m)
print(f"\nipopt:  {result_ipopt.solver.termination_condition}")
print(f"  obj = {value(m.obj):.6e}")
print(f"  x = {value(m.x):.10f}, y = {value(m.y):.10f}")

L-BFGS: start, n=2, f=2.420000e1, ||g||=2.156000e2
L-BFGS iter 0: f=4.128138e0, ||g||=2.156000e2, alpha=7.89e-4
L-BFGS iter 34: converged (optimal), f=5.395054e-21, ||g||=2.841787e-9
ripopt: L-BFGS solved unconstrained problem (Optimal, obj=5.395054e-21)
ripopt 0.2.0: Optimal after 34 iterations
Objective: 5.395054095362468e-21
ripopt: optimal
  obj = 5.395054e-21
  x = 1.0000000000, y = 1.0000000000

ipopt:  optimal
  obj = 3.743976e-21
  x = 0.9999999999, y = 0.9999999999


## Example 2: HS071 (Constrained NLP)

Minimize $x_1 x_4 (x_1 + x_2 + x_3) + x_3$

Subject to:
- $x_1 x_2 x_3 x_4 \geq 25$
- $x_1^2 + x_2^2 + x_3^2 + x_4^2 = 40$
- $1 \leq x_i \leq 5$

Known solution: $f^* \approx 17.0140$

In [8]:
def build_hs071():
    m = ConcreteModel()
    m.x1 = Var(bounds=(1, 5), initialize=1.0)
    m.x2 = Var(bounds=(1, 5), initialize=5.0)
    m.x3 = Var(bounds=(1, 5), initialize=5.0)
    m.x4 = Var(bounds=(1, 5), initialize=1.0)
    m.obj = Objective(expr=m.x1*m.x4*(m.x1 + m.x2 + m.x3) + m.x3)
    m.c1 = Constraint(expr=m.x1*m.x2*m.x3*m.x4 >= 25)
    m.c2 = Constraint(expr=m.x1**2 + m.x2**2 + m.x3**2 + m.x4**2 == 40)
    return m

# Solve with ripopt
m = build_hs071()
result = SolverFactory('ripopt').solve(m, tee=True)
print(f"ripopt: {result.solver.termination_condition}")
print(f"  obj = {value(m.obj):.6f}")
print(f"  x = [{value(m.x1):.6f}, {value(m.x2):.6f}, {value(m.x3):.6f}, {value(m.x4):.6f}]")

# Solve with ipopt
m = build_hs071()
result = SolverFactory('ipopt').solve(m)
print(f"\nipopt:  {result.solver.termination_condition}")
print(f"  obj = {value(m.obj):.6f}")
print(f"  x = [{value(m.x1):.6f}, {value(m.x2):.6f}, {value(m.x3):.6f}, {value(m.x4):.6f}]")

iter      objective     inf_pr     inf_du      compl         mu  alpha_p  alpha_d
   0    1.6200899e1     1.18e1     0.00e0    1.00e-1    1.00e-1   0.00e0   0.00e0
   1    1.4289717e1     5.48e0     0.00e0    1.00e-1    7.45e-3   1.00e0  1.67e-2
   2    1.7361221e1    1.40e-1     0.00e0    9.42e-2    6.62e-3   1.00e0  2.86e-2
   3    1.6895054e1    3.11e-1     0.00e0    3.24e-2    2.10e-3   1.00e0  7.16e-1
   4    1.6915395e1    2.22e-1     0.00e0    7.08e-3    3.05e-4  3.16e-1  9.49e-1
   5    1.6988481e1    4.31e-2     0.00e0    1.76e-3    4.75e-5   1.00e0   1.00e0
   6    1.7008042e1    9.76e-3     0.00e0    6.38e-5    4.38e-6  7.76e-1   1.00e0
   7    1.7014000e1    3.74e-5     0.00e0    4.95e-6    4.33e-7   1.00e0   1.00e0
   8    1.7014010e1    1.02e-5     0.00e0    4.33e-7    3.79e-8  7.27e-1   1.00e0
   9    1.7014017e1   8.48e-10     0.00e0    3.79e-8    3.63e-9   1.00e0   1.00e0
  10    1.7014017e1   2.20e-10     0.00e0    3.63e-9   3.18e-10  7.41e-1   1.00e0

Phase breakdown

## Example 3: Quadratic Program with Inequality Constraints

Minimize $(x - 1)^2 + (y - 2)^2$

Subject to: $x + y \leq 2$, $x, y \geq 0$

Known solution: $(x^*, y^*) = (0.5, 1.5)$, $f^* = 0.5$

In [10]:
def build_qp():
    m = ConcreteModel()
    m.x = Var(bounds=(0, None), initialize=0.5)
    m.y = Var(bounds=(0, None), initialize=0.5)
    m.obj = Objective(expr=(m.x - 1)**2 + (m.y - 2)**2)
    m.c1 = Constraint(expr=m.x + m.y <= 2)
    return m

m = build_qp()
result = SolverFactory('ripopt').solve(m, tee=True)
print(f"ripopt: {result.solver.termination_condition}")
print(f"  obj = {value(m.obj):.6f}, x = {value(m.x):.6f}, y = {value(m.y):.6f}")

m = build_qp()
result = SolverFactory('ipopt').solve(m)
print(f"ipopt:  {result.solver.termination_condition}")
print(f"  obj = {value(m.obj):.6f}, x = {value(m.x):.6f}, y = {value(m.y):.6f}")

ripopt: Detected 1/1 linear constraints (Hessian contribution skipped)
iter      objective     inf_pr     inf_du      compl         mu  alpha_p  alpha_d
   0    2.5000000e0     0.00e0     3.00e0    1.00e-1    1.00e-1   0.00e0   0.00e0
   1   4.0516105e-2    7.69e-1    2.92e-1    1.25e-1    6.41e-3   1.00e0  3.80e-1
   2   5.0430457e-1     0.00e0     0.00e0    3.43e-2    2.00e-3   1.00e0   1.00e0
   3   5.0045354e-1    6.82e-5     0.00e0    3.68e-3    2.80e-4   1.00e0   1.00e0
   4   5.0000512e-1     0.00e0     0.00e0    3.73e-4    3.18e-5   1.00e0   1.00e0
   5   5.0000003e-1    1.60e-8     0.00e0    3.28e-5    3.22e-6   1.00e0   1.00e0
   6   5.0000000e-1     0.00e0     0.00e0    3.23e-6    3.22e-7   1.00e0   1.00e0
   7   5.0000000e-1   1.53e-12     0.00e0    3.23e-7    3.23e-8   1.00e0   1.00e0
   8   5.0000000e-1   1.54e-12     0.00e0    3.23e-8    3.23e-9   1.00e0   1.00e0
   9   5.0000000e-1   1.54e-12     0.00e0    3.23e-9   3.23e-10   1.00e0   1.00e0

Phase breakdown (10 iterat

## Example 4: Linear Program

Maximize $3x + 5y$

Subject to: $x + 2y \leq 14$, $3x + 2y \leq 18$, $x, y \geq 0$

In [11]:
def build_lp():
    m = ConcreteModel()
    m.x = Var(bounds=(0, 10))
    m.y = Var(bounds=(0, 10))
    m.obj = Objective(expr=3*m.x + 5*m.y, sense=maximize)
    m.c1 = Constraint(expr=m.x + 2*m.y <= 14)
    m.c2 = Constraint(expr=3*m.x + 2*m.y <= 18)
    return m

m = build_lp()
result = SolverFactory('ripopt').solve(m, tee=True)
print(f"ripopt: {result.solver.termination_condition}")
print(f"  obj = {value(m.obj):.6f}, x = {value(m.x):.6f}, y = {value(m.y):.6f}")

m = build_lp()
result = SolverFactory('ipopt').solve(m)
print(f"ipopt:  {result.solver.termination_condition}")
print(f"  obj = {value(m.obj):.6f}, x = {value(m.x):.6f}, y = {value(m.y):.6f}")

ripopt: Detected 2/2 linear constraints (Hessian contribution skipped)
iter      objective     inf_pr     inf_du      compl         mu  alpha_p  alpha_d
   0  -8.0000000e-2     0.00e0     0.00e0    1.00e-1    1.00e-1   0.00e0   0.00e0
   1  -2.7363955e-1     0.00e0     0.00e0    9.99e-2    5.82e-3   1.00e0  3.97e-1
   2   -9.8979030e0     0.00e0     0.00e0    1.46e-1    8.13e-3   1.00e0  1.91e-2
   3   -3.0022871e1     0.00e0     0.00e0    1.47e-1    7.33e-3  2.83e-3  1.31e-3
   4   -3.5105616e1    5.37e-3     0.00e0     1.54e0    4.00e-2  2.56e-3  4.18e-1
   5   -3.6860821e1     3.44e0     0.00e0    1.62e-1    6.13e-3   1.00e0  6.40e-1
   6   -3.6010872e1    4.83e-3     0.00e0    4.86e-2    1.61e-3   1.00e0  8.85e-1
   7   -3.5998938e1     0.00e0     0.00e0    1.61e-3    1.61e-4   1.00e0   1.00e0
   8   -3.5999999e1    8.99e-5     0.00e0    1.61e-4    1.61e-5   1.00e0   1.00e0
   9   -3.5999984e1     0.00e0     0.00e0    1.61e-5    1.61e-6   1.00e0   1.00e0
  10   -3.6000000e1    8.92

## Example 5: Nonlinear Regression

Fit $y = a \cdot e^{b \cdot x}$ to noisy data using least squares.

In [12]:
import numpy as np

# Generate synthetic data
np.random.seed(42)
x_data = np.linspace(0, 2, 10)
y_data = 2.5 * np.exp(0.8 * x_data) + np.random.normal(0, 0.3, len(x_data))

def build_regression():
    m = ConcreteModel()
    m.a = Var(initialize=1.0)
    m.b = Var(initialize=0.5)
    m.obj = Objective(
        expr=sum((y_data[i] - m.a * exp(m.b * x_data[i]))**2 for i in range(len(x_data)))
    )
    return m

m = build_regression()
result = SolverFactory('ripopt').solve(m, tee=True)
print(f"ripopt: {result.solver.termination_condition}")
print(f"  a = {value(m.a):.6f}, b = {value(m.b):.6f}  (true: a=2.5, b=0.8)")
print(f"  residual = {value(m.obj):.6f}")

m = build_regression()
result = SolverFactory('ipopt').solve(m)
print(f"\nipopt:  {result.solver.termination_condition}")
print(f"  a = {value(m.a):.6f}, b = {value(m.b):.6f}  (true: a=2.5, b=0.8)")
print(f"  residual = {value(m.obj):.6f}")

L-BFGS: start, n=2, f=2.897359e2, ||g||=2.859073e2
L-BFGS iter 0: f=1.415781e1, ||g||=2.859073e2, alpha=2.27e-3
L-BFGS iter 14: step too small (3.04e-16), acceptable
ripopt: L-BFGS solved unconstrained problem (Acceptable, obj=4.153749e-1)
ripopt 0.2.0: Acceptable after 14 iterations
Objective: 4.153749340660062e-1
model.name="unknown";
    - termination condition: optimal
    - message from solver: ripopt 0.2.0\x3a Solved To Acceptable Level
ripopt: optimal
  a = 2.626218, b = 0.778529  (true: a=2.5, b=0.8)
  residual = 0.415375

ipopt:  optimal
  a = 2.626218, b = 0.778529  (true: a=2.5, b=0.8)
  residual = 0.415375
